# Coffee Data Cleaning for a Yunnan Overseas-Entry Strategy

## Goal
Prepare four separate datasets for market screening, channel investigation, and packaging decisions. Keep Coffee and RTD Coffee separate. The outputs support further research; they do not establish which country is profitable or feasible to enter.

**Result:** 8,675 data rows retained; only 26 verified blank/footer rows excluded. Original numbers and 6,260 unavailable annual observations are preserved. Four CSVs remain separate.

Read [plan.md](plan.md) for column rules and the October 2 implementation revision, [cleaning-report.md](cleaning-report.md) for the individual checklist and limitations, and [README.md](README.md) for access and run instructions. The initial Market Sizes draft preceded the plan; this four-table implementation follows the revised plan. All files are in English.

## Setup
Choose the **coffee market** kernel, restart it, and run all cells from the repository root. The notebook calls the same readable functions as the command-line workflow in [clean_coffee.py](clean_coffee.py). This prevents the notebook and script from applying different rules. Original XLS files are never overwritten; rerunning regenerates the CSVs and audit files.

**Assumptions:** annual `-` means unavailable for computation, not zero; its precise Passport meaning is unverified. Countries, regions, and World overlap. Channel and packaging hierarchies also overlap. No imputation, cross-table merge, currency conversion, or summing across hierarchy levels is performed.

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "clean_coffee.py").exists():
    raise RuntimeError("Open this notebook from the project root. Check Path.cwd() before changing paths.")
from clean_coffee import clean_dataset, to_long, country_screen, save_csv
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 55)
print("Working directory:", ROOT)
print("Python executable:", sys.executable)
results = {}

Working directory: /Users/wendysun/Projects/cpg-global-entry-strategy
Python executable: /Users/wendysun/Projects/cpg-global-entry-strategy/.venv/bin/python


## A. Market Sizes

**Purpose:** Which observed foreign markets have substantial retail coffee volume and historical growth?

Keep sales volume and monetary sales as different measures. Preserve Tonnes, million litres, and original currencies/scales. `Current Constant = -` remains a descriptor on volume rows. India's RTD annual dashes become missing numeric observations, while both records remain. No currency conversion is justified in this cleaning pass.

The next cell loads the original file, checks the footer, trims labels, converts year values, validates keys, flags unusual observations, and exports a separate CSV. Detailed checks use every record; displays below are intentionally short.

In [2]:
market_sizes_result = clean_dataset("market_sizes")
results["market_sizes"] = market_sizes_result
market_sizes_clean = market_sizes_result["clean"]
summary_fields = ["imported_rows", "cleaned_rows", "excluded_rows", "missing_values", "text_cells_trimmed", "flagged_observations"]
display(pd.Series({key: market_sizes_result["summary"][key] for key in summary_fields}, name="Result").to_frame())
display(market_sizes_clean.head(3))

,Result
imported_rows,79
cleaned_rows,72
excluded_rows,7
missing_values,22
text_cells_trimmed,0
flagged_observations,7


,Geography,Category,Data Type,Unit,Current Constant,2015,...,2020,2021,2022,2023,2024,2025
0,World,Coffee,Retail Volume,Tonnes,-,5485017.6,...,6163793.8,6211134.5,6084333.8,6123042.7,6148465.5,6175791.8
1,World,Coffee,Retail Value RSP,USD million,Current Prices,79774.3,...,92552.5,98356.5,102967.6,111017.1,116219.3,125307.9
2,World,RTD Coffee,Off-trade Volume,million litres,-,4884.7,...,5411.7,5593.4,5726.3,5792.3,5722.3,5667.1


### Missing values and text changes

Compare original blank/dash counts with missing numeric values after conversion. Unexpected conversion failures stop the run. Text-change examples show actual source row numbers; no data row is removed for missing annual values.

In [3]:
display(market_sizes_result["missingness"])
print("Text changes:", len(market_sizes_result["text_changes"]))
display(market_sizes_result["text_changes"].head(3))

,Year,Before dtype,After dtype,Original blanks,Original dashes,Missing after,Valid after,Conversion failures
0,2015,object,float64,0,2,2,70,0
1,2016,object,float64,0,2,2,70,0
2,2017,object,float64,0,2,2,70,0
3,2018,object,float64,0,2,2,70,0
4,2019,object,float64,0,2,2,70,0
5,2020,object,float64,0,2,2,70,0
6,2021,object,float64,0,2,2,70,0
7,2022,object,float64,0,2,2,70,0
8,2023,object,float64,0,2,2,70,0
9,2024,object,float64,0,2,2,70,0


Text changes: 0


,Source Excel Row,Column,Before,After


### Check records and retained flags

Five planned checks compare source, expected, and cleaned values. The separate validator also checks every original number, missing observation, and descriptor directly against Excel. Full flag lists and compatible-group numeric distributions are saved under `data/quality/`; flags need business review before interpretation.

In [4]:
display(market_sizes_result["checks"][["Source Excel Row", "Year", "Original Value", "Expected Value", "Actual Value", "Match"]])
display(market_sizes_result["flags"].head(3))

,Source Excel Row,Year,Original Value,Expected Value,Actual Value,Match
0,15,2025,61314.5,61314.5,61314.5,True
1,24,2025,461.1,461.1,461.1,True
2,21,2015,-,NaN,NaN,True
3,22,2025,-,NaN,NaN,True
4,7,2025,6175791.8,6175791.8,6175791.8,True


,Source Excel Row,Geography,Category,Data Type,Unit,Current Constant,Year,Value,Previous Value,Reason,Action
0,33,Australasia,RTD Coffee,Off-trade Volume,million litres,-,2019,5.4,3.5,Adjacent relative change greater than 50 percent,Retained; investigate before interpretation
1,37,Australia,RTD Coffee,Off-trade Volume,million litres,-,2019,5.2,3.4,Adjacent relative change greater than 50 percent,Retained; investigate before interpretation
2,38,Australia,RTD Coffee,Off-trade Value RSP,AUD million,Current Prices,2019,51.1,32.8,Adjacent relative change greater than 50 percent,Retained; investigate before interpretation


## B. Retail Channels

**Purpose:** Which retail routes deserve investigation in each candidate country?

The workbook title explicitly says `% breakdown`. Add `Unit = %` without rescaling. Keep all 24 outlet labels, including totals and parents. Do not sum all outlets or average country shares into a regional share. Changes greater than 10 percentage points are review flags, not automatic errors.

The next cell loads the original file, checks the footer, trims labels, converts year values, validates keys, flags unusual observations, and exports a separate CSV. Detailed checks use every record; displays below are intentionally short.

In [5]:
retail_channels_result = clean_dataset("retail_channels")
results["retail_channels"] = retail_channels_result
retail_channels_clean = retail_channels_result["clean"]
summary_fields = ["imported_rows", "cleaned_rows", "excluded_rows", "missing_values", "text_cells_trimmed", "flagged_observations"]
display(pd.Series({key: retail_channels_result["summary"][key] for key in summary_fields}, name="Result").to_frame())
display(retail_channels_clean.head(3))

,Result
imported_rows,871
cleaned_rows,864
excluded_rows,7
missing_values,3384
text_cells_trimmed,0
flagged_observations,2


,Geography,Category,Outlet Type,Data Type,Unit,2015,...,2020,2021,2022,2023,2024,2025
0,World,Coffee,Retail Channels,Retail Volume,%,100.0,...,100.0,100.0,100.0,100.0,100.0,100.0
1,World,Coffee,Retail Offline,Retail Volume,%,97.8,...,94.4,94.0,93.4,92.9,91.8,91.4
2,World,Coffee,Grocery Retailers,Retail Volume,%,95.1,...,92.1,91.6,91.1,90.7,89.6,89.2


### Missing values and text changes

Compare original blank/dash counts with missing numeric values after conversion. Unexpected conversion failures stop the run. Text-change examples show actual source row numbers; no data row is removed for missing annual values.

In [6]:
display(retail_channels_result["missingness"])
print("Text changes:", len(retail_channels_result["text_changes"]))
display(retail_channels_result["text_changes"].head(3))

,Year,Before dtype,After dtype,Original blanks,Original dashes,Missing after,Valid after,Conversion failures
0,2015,object,float64,0,315,315,549,0
1,2016,object,float64,0,314,314,550,0
2,2017,object,float64,0,310,310,554,0
3,2018,object,float64,0,314,314,550,0
4,2019,object,float64,0,310,310,554,0
5,2020,object,float64,0,309,309,555,0
6,2021,object,float64,0,308,308,556,0
7,2022,object,float64,0,301,301,563,0
8,2023,object,float64,0,304,304,560,0
9,2024,object,float64,0,298,298,566,0


Text changes: 0


,Source Excel Row,Column,Before,After


### Check records and retained flags

Five planned checks compare source, expected, and cleaned values. The separate validator also checks every original number, missing observation, and descriptor directly against Excel. Full flag lists and compatible-group numeric distributions are saved under `data/quality/`; flags need business review before interpretation.

In [7]:
display(retail_channels_result["checks"][["Source Excel Row", "Year", "Original Value", "Expected Value", "Actual Value", "Match"]])
display(retail_channels_result["flags"].head(3))

,Source Excel Row,Year,Original Value,Expected Value,Actual Value,Match
0,7,2015,100,100.0,100.0,True
1,8,2015,97.8,97.8,97.8,True
2,29,2015,2.2,2.2,2.2,True
3,21,2015,-,NaN,NaN,True
4,30,2015,100,100.0,100.0,True


,Source Excel Row,Geography,Category,Outlet Type,Data Type,Year,Value,Previous Value,Reason,Action
0,133,China,RTD Coffee,Supermarkets,Off-trade Volume,2020,19.1,36.3,Adjacent change greater than 10 percentage points,Retained; investigate before interpretation
1,138,China,RTD Coffee,Small Local Grocers,Off-trade Volume,2020,46.3,33.6,Adjacent change greater than 10 percentage points,Retained; investigate before interpretation


## C. Pack Type

**Purpose:** Which packaging formats are represented in each country, and how have their unit volumes changed?

Annual values count **million packaging units**, not tonnes or sales revenue. Trim trailing whitespace in `PET Jars ` while keeping meaningful labels such as `Other Packaging`. Preserve parents, children, and totals; they cannot all be added together. Large changes and movements from zero are retained for review.

The next cell loads the original file, checks the footer, trims labels, converts year values, validates keys, flags unusual observations, and exports a separate CSV. Detailed checks use every record; displays below are intentionally short.

In [8]:
pack_type_result = clean_dataset("pack_type")
results["pack_type"] = pack_type_result
pack_type_clean = pack_type_result["clean"]
summary_fields = ["imported_rows", "cleaned_rows", "excluded_rows", "missing_values", "text_cells_trimmed", "flagged_observations"]
display(pd.Series({key: pack_type_result["summary"][key] for key in summary_fields}, name="Result").to_frame())
display(pack_type_clean.head(3))

,Result
imported_rows,536
cleaned_rows,530
excluded_rows,6
missing_values,96
text_cells_trimmed,9
flagged_observations,100


,Geography,Category,Packaging Class,Pack Type,Data Type,Unit,...,2020,2021,2022,2023,2024,2025
0,Western Europe,Coffee,Total,Total Packaging,Retail/off-trade Unit Volume,million units,...,12080.3,12352.1,12052.9,12023.0,12135.5,12217.6
1,Eastern Europe,Coffee,Total,Total Packaging,Retail/off-trade Unit Volume,million units,...,4919.0,4973.3,4647.0,4830.5,4740.6,4701.0
2,North America,Coffee,Total,Total Packaging,Retail/off-trade Unit Volume,million units,...,3497.4,3391.8,3185.6,3241.2,3279.0,3314.5


### Missing values and text changes

Compare original blank/dash counts with missing numeric values after conversion. Unexpected conversion failures stop the run. Text-change examples show actual source row numbers; no data row is removed for missing annual values.

In [9]:
display(pack_type_result["missingness"])
print("Text changes:", len(pack_type_result["text_changes"]))
display(pack_type_result["text_changes"].head(3))

,Year,Before dtype,After dtype,Original blanks,Original dashes,Missing after,Valid after,Conversion failures
0,2015,object,float64,0,14,14,516,0
1,2016,object,float64,0,13,13,517,0
2,2017,object,float64,0,11,11,519,0
3,2018,object,float64,0,8,8,522,0
4,2019,object,float64,0,8,8,522,0
5,2020,object,float64,0,8,8,522,0
6,2021,object,float64,0,5,5,525,0
7,2022,object,float64,0,5,5,525,0
8,2023,object,float64,0,8,8,522,0
9,2024,object,float64,0,8,8,522,0


Text changes: 9


,Source Excel Row,Column,Before,After
0,297,Pack Type,PET Jars,PET Jars
1,298,Pack Type,PET Jars,PET Jars
2,299,Pack Type,PET Jars,PET Jars


### Check records and retained flags

Five planned checks compare source, expected, and cleaned values. The separate validator also checks every original number, missing observation, and descriptor directly against Excel. Full flag lists and compatible-group numeric distributions are saved under `data/quality/`; flags need business review before interpretation.

In [10]:
display(pack_type_result["checks"][["Source Excel Row", "Year", "Original Value", "Expected Value", "Actual Value", "Match"]])
display(pack_type_result["flags"].head(3))

,Source Excel Row,Year,Original Value,Expected Value,Actual Value,Match
0,7,2015,9598.9,9598.9,9598.9,True
1,45,2015,-,NaN,NaN,True
2,45,2025,0.1,0.1,0.1,True
3,297,2025,24.2,24.2,24.2,True
4,536,2025,10,10.0,10.0,True


,Source Excel Row,Geography,Category,Packaging Class,Pack Type,Data Type,Unit,Year,Value,Previous Value,Reason,Action
0,83,World,Coffee,Total,Flexible Paper/Plastic,Retail/off-trade Unit Volume,million units,2016,118.4,54.6,Adjacent relative change greater than 50 percent,Retained; investigate before interpretation
1,85,Western Europe,Coffee,Total,Flexible Paper/Plastic,Retail/off-trade Unit Volume,million units,2016,118.4,54.6,Adjacent relative change greater than 50 percent,Retained; investigate before interpretation
2,292,China,Coffee,Total,Rigid Plastic,Retail/off-trade Unit Volume,million units,2016,0.6,0.3,Adjacent relative change greater than 50 percent,Retained; investigate before interpretation


## D. Pack Size

**Purpose:** Which package sizes deserve investigation within a country and product category?

Preserve original size labels, including `Total`. Annual values still measure **million packaging units**; `100 g` and `250 ml` describe package sizes. The optional long view separates size number and unit for sorting, with missing numeric size for `Total`. It does not turn packaging counts into product weight or litres.

The next cell loads the original file, checks the footer, trims labels, converts year values, validates keys, flags unusual observations, and exports a separate CSV. Detailed checks use every record; displays below are intentionally short.

In [11]:
pack_size_result = clean_dataset("pack_size")
results["pack_size"] = pack_size_result
pack_size_clean = pack_size_result["clean"]
summary_fields = ["imported_rows", "cleaned_rows", "excluded_rows", "missing_values", "text_cells_trimmed", "flagged_observations"]
display(pd.Series({key: pack_size_result["summary"][key] for key in summary_fields}, name="Result").to_frame())
display(pack_size_clean.head(3))

,Result
imported_rows,7215
cleaned_rows,7209
excluded_rows,6
missing_values,2758
text_cells_trimmed,56
flagged_observations,1845


,Geography,Category,Packaging Class,Pack Type,Pack Size,Data Type,...,2020,2021,2022,2023,2024,2025
0,Eastern Europe,Coffee,Total,Total Packaging,10 g,Retail/off-trade Unit Volume,...,7.0,6.2,4.7,3.9,2.5,0.8
1,North America,Coffee,Total,Total Packaging,10 g,Retail/off-trade Unit Volume,...,40.6,37.4,35.8,37.1,43.0,40.8
2,Latin America,Coffee,Total,Total Packaging,10 g,Retail/off-trade Unit Volume,...,91.5,116.2,119.5,125.1,133.8,133.2


### Missing values and text changes

Compare original blank/dash counts with missing numeric values after conversion. Unexpected conversion failures stop the run. Text-change examples show actual source row numbers; no data row is removed for missing annual values.

In [12]:
display(pack_size_result["missingness"])
print("Text changes:", len(pack_size_result["text_changes"]))
display(pack_size_result["text_changes"].head(3))

,Year,Before dtype,After dtype,Original blanks,Original dashes,Missing after,Valid after,Conversion failures
0,2015,object,float64,0,411,411,6798,0
1,2016,object,float64,0,329,329,6880,0
2,2017,object,float64,0,303,303,6906,0
3,2018,object,float64,0,264,264,6945,0
4,2019,object,float64,0,218,218,6991,0
5,2020,object,float64,0,216,216,6993,0
6,2021,object,float64,0,197,197,7012,0
7,2022,object,float64,0,197,197,7012,0
8,2023,object,float64,0,207,207,7002,0
9,2024,object,float64,0,207,207,7002,0


Text changes: 56


,Source Excel Row,Column,Before,After
0,5468,Pack Type,PET Jars,PET Jars
1,5469,Pack Type,PET Jars,PET Jars
2,5470,Pack Type,PET Jars,PET Jars


### Check records and retained flags

Five planned checks compare source, expected, and cleaned values. The separate validator also checks every original number, missing observation, and descriptor directly against Excel. Full flag lists and compatible-group numeric distributions are saved under `data/quality/`; flags need business review before interpretation.

In [13]:
display(pack_size_result["checks"][["Source Excel Row", "Year", "Original Value", "Expected Value", "Actual Value", "Match"]])
display(pack_size_result["flags"].head(3))

,Source Excel Row,Year,Original Value,Expected Value,Actual Value,Match
0,14,2025,3774.2,3774.2,3774.2,True
1,41,2015,-,NaN,NaN,True
2,916,2025,3314.5,3314.5,3314.5,True
3,5468,2025,0.1,0.1,0.1,True
4,6021,2025,2508.1,2508.1,2508.1,True


,Source Excel Row,Geography,Category,Packaging Class,Pack Type,Pack Size,...,Unit,Year,Value,Previous Value,Reason,Action
0,76,Asia Pacific,Coffee,Total,Total Packaging,112 g,...,million units,2016,20.0,10.7,Adjacent relative change greater than 50 percent,Retained; investigate before interpretation
1,80,Japan,Coffee,Total,Total Packaging,112 g,...,million units,2016,19.6,10.4,Adjacent relative change greater than 50 percent,Retained; investigate before interpretation
2,108,China,Coffee,Total,Total Packaging,118 g,...,million units,2016,0.9,0.5,Adjacent relative change greater than 50 percent,Retained; investigate before interpretation


## Use the cleaned data for the project

The optional long view creates `Year` and `Value` columns and explicit geographic levels. It retains missing observations and source categories. Use a country filter for country comparisons; China is the domestic benchmark. This example previews parsed size fields without merging datasets.

In [14]:
pack_size_long = to_long(pack_size_clean, "pack_size")
display(pack_size_long[["Geography", "Geography Level", "Market Role", "Pack Size", "Size Value", "Size Unit", "Is Size Total"]].drop_duplicates().head(8))
assert len(pack_size_long) == len(pack_size_clean) * 11

,Geography,Geography Level,Market Role,Pack Size,Size Value,Size Unit,Is Size Total
0,Eastern Europe,Region,Context only,10 g,10,g,False
1,North America,Region,Context only,10 g,10,g,False
2,Latin America,Region,Context only,10 g,10,g,False
3,Asia Pacific,Region,Context only,10 g,10,g,False
4,Japan,Country,Foreign candidate,10 g,10,g,False
5,USA,Country,Foreign candidate,10 g,10,g,False
6,World,World,Context only,10 g,10,g,False
7,World,World,Context only,100 g,100,g,False


### Descriptive market screen

Compare Coffee volume only with Coffee volume, and RTD only with RTD. CAGR uses 2015 and 2025, a ten-year interval, and requires a positive starting value and both endpoints. `Available Years` reports coverage; CAGR is an endpoint measure and does not describe the year-by-year path. India's unavailable RTD growth remains missing.

The table is alphabetical, not a recommendation ranking. These broad categories do not isolate Yunnan coffee, specialty beans, or the company's addressable segment. No values from the old presentation dashboard are used.

In [15]:
screen = country_screen(market_sizes_clean)
save_csv(screen, ROOT / "data/coffee_country_screening.csv")
display(screen.loc[screen.Category.eq("Coffee"), ["Geography", "Unit", "2015", "2025", "Market Role", "Available Years", "CAGR 2015-2025 (%)"]].round(2))
display(screen.groupby("Category")["CAGR 2015-2025 (%)"].agg(candidate_and_benchmark_rows="size", valid_growth_estimates="count"))

,Geography,Unit,2015,2025,Market Role,Available Years,CAGR 2015-2025 (%)
0,Australia,Tonnes,34573.7,40315.7,Foreign candidate,11,1.55
1,Brazil,Tonnes,707374.4,830985.4,Foreign candidate,11,1.62
2,China,Tonnes,63458.7,61314.5,Domestic benchmark,11,-0.34
3,France,Tonnes,181134.9,178909.5,Foreign candidate,11,-0.12
4,Germany,Tonnes,381919.7,405072.7,Foreign candidate,11,0.59
5,India,Tonnes,55723.8,104240.0,Foreign candidate,11,6.46
6,Japan,Tonnes,109749.0,97775.2,Foreign candidate,11,-1.15
7,Thailand,Tonnes,67308.4,89133.0,Foreign candidate,11,2.85
8,USA,Tonnes,782346.6,735187.2,Foreign candidate,11,-0.62
9,United Kingdom,Tonnes,77903.9,74710.0,Foreign candidate,11,-0.42


,candidate_and_benchmark_rows,valid_growth_estimates
Category,,
Coffee,10,10
RTD Coffee,10,9


## Full-data verification and fresh-process rerun

Save the combined evidence, then run `validate_cleaning.py --reproduce`. It reads the XLS cells independently with xlrd, checks every descriptor and annual observation, validates the growth calculations, and launches a fresh Python process to recreate the data and audit outputs. Byte-identical outputs and unchanged originals are required. This does not resolve source definitions or replace human review.

In [16]:
summary = pd.DataFrame([result["summary"] for result in results.values()])
save_csv(summary, ROOT / "data/quality/summary.csv")
save_csv(pd.concat([result["checks"] for result in results.values()], ignore_index=True), ROOT / "data/quality/record_checks.csv")
completed = subprocess.run([sys.executable, str(ROOT / "validate_cleaning.py"), "--reproduce"], cwd=ROOT, capture_output=True, text=True)
if completed.returncode:
    raise RuntimeError(completed.stdout + completed.stderr)
verification = json.loads((ROOT / "data/quality/independent_validation.json").read_text())
display(pd.DataFrame(verification["independent_excel_cell_checks"]))
print("Fresh-process outputs identical:", verification["fresh_process_outputs_identical"])
display(summary[["dataset", "cleaned_rows", "missing_values", "conversion_failures", "key_after", "record_checks_passed"]])

,dataset,rows_verified,numeric_cells_verified,missing_cells_verified,descriptor_cells_verified
0,market_sizes,72,770,22,360
1,retail_channels,864,6120,3384,3456
2,pack_type,530,5734,96,3180
3,pack_size,7209,76541,2758,50463


Fresh-process outputs identical: True


,dataset,cleaned_rows,missing_values,conversion_failures,key_after,record_checks_passed
0,market_sizes,72,22,0,0,5
1,retail_channels,864,3384,0,0,5
2,pack_type,530,96,0,0,5
3,pack_size,7209,2758,0,0,5


## What still needs a human decision

- Confirm the company's product, budget, capacity, and price tier. Keep RTD separate until its relevance is decided.
- Confirm Passport's dash meaning and channel/packaging hierarchy definitions. Retained large changes are source-verified, but their causes are not established.
- Only nine foreign countries are observed. Missing countries are outside this export, not unattractive or zero-sized markets.
- Obtain competition, import-access, cost/margin, logistics, and customer evidence before recommending entry. Current-price local-currency rows cannot establish comparable real growth or cross-country spending.
- Review `cleaning-report.md` before submitting the individual PDF on Gradescope. The updated assignment does not require a group submission. The 40 rows in `submission/samples/` are real cleaned-data subsets with source row references; full validation uses the entire local dataset.

**AI assistance:** Codex implemented and verified the documented rules. The student remains responsible for understanding decisions, resolving uncertainties, and reviewing the final submission.